# K-KODE Engine v55.0 — Interactive Demo

This notebook runs the real K-KODE pipeline end-to-end on a synthetic cohort and shows the results visually — no need to read the code to see what it does.

**What this demonstrates:**
- Floor-censored data (fast progressors who cross the measurement floor) is **kept and modeled**, not dropped
- Four candidate decay shapes compete per patient via AICc (Linear, Square-Root, Log-Exponential, Power-Law)
- Population-level decay rate and clinical-trial sample size are computed
- A plot showing each patient's actual data, fitted curve, and which points were floor-censored


In [ ]:
# 1. Install dependencies & clone repository
!pip install -q pandas numpy scipy statsmodels matplotlib
!rm -rf /content/KKODE-BIO
!git clone https://github.com/EAI-BIO/KKODE-BIO.git
import sys
if '/content/KKODE-BIO' not in sys.path:
    sys.path.append('/content/KKODE-BIO')

In [ ]:
# 2. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kkode_engine import KKodeApexEngine

## 3. Build a synthetic cohort

8 patients: 5 typical progressors, and 3 fast progressors who cross the measurement floor —
the exact case K-KODE is built to keep in the analysis instead of dropping.

In [ ]:
rng = np.random.default_rng(7)
MEASUREMENT_FLOOR = 0.10
visit_years = [0.0, 1.0, 2.0, 3.0, 4.0]
rows = []

# 5 typical progressors
for pid in range(1, 6):
    baseline = rng.uniform(2.0, 3.0)
    rate = rng.uniform(0.12, 0.22)
    for t in visit_years:
        val = baseline * np.exp(-rate * t) + rng.normal(0, 0.05)
        rows.append({"patient_id": f"P{pid:02d}", "eye": "OD",
                     "visit_date": pd.Timestamp("2019-01-01") + pd.Timedelta(days=int(t * 365.25)),
                     "ez_width_mm": max(val, 0.01)})

# 3 fast progressors who cross the measurement floor
for pid in range(6, 9):
    baseline = rng.uniform(2.0, 2.6)
    rate = rng.uniform(0.75, 0.95)
    for t in visit_years:
        val = baseline * np.exp(-rate * t) + rng.normal(0, 0.03)
        rows.append({"patient_id": f"P{pid:02d}", "eye": "OS",
                     "visit_date": pd.Timestamp("2019-01-01") + pd.Timedelta(days=int(t * 365.25)),
                     "ez_width_mm": max(val, 0.0)})

df = pd.DataFrame(rows)
print(f"Synthetic cohort: {df['patient_id'].nunique()} patients, {len(df)} visit rows")
df.head()

## 4. Run the full K-KODE pipeline

In [ ]:
engine = KKodeApexEngine(
    data_source=df,
    endpoint_column="ez_width_mm",
    eye_column="eye",
    measurement_floor=MEASUREMENT_FLOOR,
)

engine.clean_and_transform()
engine.run_model_competition()
primary_model = engine.model_selection_results.get("overall_best_supported_model", "Log-Exponential")
print(f"Primary model selected: {primary_model}")

engine.run_per_patient_decay(model=primary_model)
engine.fit_mixed_effects_nlme(model=primary_model)
engine.compute_closed_form_sample_size(target_power=0.80, alpha=0.05, therapeutic_efficacy=0.30)

print(engine.generate_report())

## 5. Visualize the fits

Each color is one patient. Dots are actual visits; 'x' markers are floor-censored visits —
kept and modeled via Tobit-style MLE, not dropped. Lines are each patient's fitted decay curve.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(engine.unique_group_ids())))

for gid, color in zip(engine.unique_group_ids(), colors):
    group = engine.clean_df[engine.clean_df["group_id"] == gid].sort_values("years_from_baseline")
    t = group["years_from_baseline"].values
    y = group["ez_width_mm"].values
    is_censored = y <= MEASUREMENT_FLOOR

    ax.scatter(t[~is_censored], y[~is_censored], color=color, s=35)
    if np.any(is_censored):
        ax.scatter(t[is_censored], y[is_censored], color=color, s=70, marker="x")

    fit = engine.per_patient_fits.get(gid)
    if isinstance(fit, dict) and "slope" in fit:
        t_line = np.linspace(t.min(), t.max(), 50)
        if fit["model"] == "Linear":
            y_line = fit["intercept"] + fit["slope"] * t_line
        elif fit["model"] == "Square-Root":
            y_line = (fit["intercept"] + fit["slope"] * t_line) ** 2
        elif fit["model"] == "Log-Exponential":
            y_line = np.exp(fit["intercept"] + fit["slope"] * t_line)
        elif fit["model"] == "Power-Law":
            offset = 1.0 / 365.25
            y_line = np.exp(fit["intercept"] + fit["slope"] * np.log(t_line + offset))
        else:
            y_line = None
        if y_line is not None:
            ax.plot(t_line, np.maximum(y_line, 0), color=color, alpha=0.6, linewidth=1.5)

ax.axhline(MEASUREMENT_FLOOR, color="gray", linestyle="--", linewidth=1, label="Measurement floor")
ax.set_xlabel("Years from baseline")
ax.set_ylabel("EZ width (mm)")
ax.set_title(f"K-KODE per-patient decay fits ({primary_model})\n'x' markers = floor-censored visits, kept and modeled (not dropped)")
ax.legend()
plt.tight_layout()
plt.show()

---
**This is a planning aid, not a finalized protocol.** All testing here uses synthetic data — see the README's Validation Status & Disclaimers section for what this version has and has not been validated against.